In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
# 打印template
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

### 查看parquet

In [ ]:
import pandas as pd
from IPython.display import display # 明确导入 display，虽然在Jupyter中通常可直接用

# 请将 'your_file.parquet' 替换为你的 Parquet 文件路径
file_path = '/root/Changling/verl-main-0521/examples/data_preprocess/train.parquet'

# 读取 Parquet 文件
try:
    df = pd.read_parquet(file_path)

    print("文件的前5行内容 (使用 display):")
    display(df.head()) # 使用 display() 来展示DataFrame

    # 如果想查看更多行，比如前10行：
    # print("\n文件的前10行内容 (使用 display):")
    # display(df.head(10))

    # 或者，如果这是Jupyter Notebook单元格中的最后一行代码，
    # 你甚至可以直接写 df.head()，它会自动被display：
    # df.head()

    print("\n数据基本信息：")
    df.info() # df.info() 通常打印到标准输出，所以这里用 print 引导一下

except FileNotFoundError:
    print(f"错误：文件 '{file_path}' 未找到。请检查文件路径是否正确。")
except Exception as e:
    print(f"读取 Parquet 文件时发生错误：{e}")

### 拆分出nq，8000train，200val

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 请将 'your_training_file.parquet' 替换为你的 Parquet 训练文件路径
input_parquet_path = 'your_training_file.parquet'
# 请将 'data_source_column_name' 替换为实际存储数据来源的列名
data_source_column = 'data_source' # 假设列名为 'data_source'
target_source_value = 'nq'         # 目标数据来源的值

# 定义输出文件路径 (可选)
output_train_path = 'nq_train_8000.parquet'
output_val_path = 'nq_val_200.parquet'

try:
    # 1. 读取整个 Parquet 文件
    df_full = pd.read_parquet(input_parquet_path)
    print(f"成功读取文件: {input_parquet_path}")
    print(f"原始数据集共有 {len(df_full)} 条记录。")

    # 2. 筛选 data_source 为 'nq' 的数据
    #    请确保 'data_source_column_name' 是你文件中正确的列名
    if data_source_column not in df_full.columns:
        print(f"错误：列 '{data_source_column}' 在文件中未找到。请检查列名。")
        # 或者在这里处理，比如退出或者尝试其他列名
    else:
        df_nq = df_full[df_full[data_source_column] == target_source_value].copy() # 使用 .copy() 避免 SettingWithCopyWarning
        print(f"筛选后，数据来源为 '{target_source_value}' 的记录共有 {len(df_nq)} 条。")

        # 3. 检查是否有足够的数据进行拆分
        required_total_nq = 8000 + 200
        if len(df_nq) < required_total_nq:
            print(f"警告：数据来源为 '{target_source_value}' 的记录不足 {required_total_nq} 条，无法按要求拆分。")
            print(f"当前只有 {len(df_nq)} 条 '{target_source_value}' 数据。请调整拆分数量或检查数据。")
            # 你可以在这里决定如何处理，比如按比例拆分，或者直接使用所有可用数据等
            # 例如，如果仍然想尽可能拆分：
            if len(df_nq) > 200: # 至少能分出验证集
                val_size_actual = min(200, int(len(df_nq) * 0.1)) # 或者取一个比例，或固定200
                if len(df_nq) - val_size_actual > 0: # 确保训练集有数据
                    df_nq_val = df_nq.sample(n=val_size_actual, random_state=42) # random_state保证可重复
                    df_nq_train = df_nq.drop(df_nq_val.index)
                    print(f"数据不足，实际拆分：训练集 {len(df_nq_train)} 条，验证集 {len(df_nq_val)} 条。")

                    # (可选) 保存拆分后的文件
                    df_nq_train.to_parquet(output_train_path, index=False)
                    df_nq_val.to_parquet(output_val_path, index=False)
                    print(f"'{target_source_value}' 训练数据已保存到: {output_train_path}")
                    print(f"'{target_source_value}' 验证数据已保存到: {output_val_path}")
                else:
                    print("数据过少，无法拆分出有效的训练集和验证集。")
            else:
                print("数据过少，无法拆分。")

        else:
            # 4. 拆分数据
            # 首先，打乱 'nq' 数据（这是一个好习惯，确保随机性）
            df_nq_shuffled = df_nq.sample(frac=1, random_state=42).reset_index(drop=True) # random_state保证可重复

            # 取前8000条作为训练集
            df_nq_train = df_nq_shuffled.iloc[:8000]

            # 取接下来的200条作为验证集
            df_nq_val = df_nq_shuffled.iloc[8000:8000+200]

            print(f"成功拆分数据来源为 '{target_source_value}' 的数据：")
            print(f"训练集: {len(df_nq_train)} 条")
            print(f"验证集: {len(df_nq_val)} 条")

            # 5. (可选) 显示拆分后数据集的前几行
            print("\n'nq' 训练集前3行:")
            print(df_nq_train.head(3))
            print("\n'nq' 验证集前3行:")
            print(df_nq_val.head(3))

            # 6. (可选) 保存拆分后的文件
            # 如果需要将拆分后的数据保存为新的 Parquet 文件：
            df_nq_train.to_parquet(output_train_path, index=False)
            df_nq_val.to_parquet(output_val_path, index=False)
            print(f"\n'{target_source_value}' 训练数据已保存到: {output_train_path}")
            print(f"'{target_source_value}' 验证数据已保存到: {output_val_path}")

            # 如果你想处理非 'nq' 的数据，可以在这里添加逻辑
            # df_other_sources = df_full[df_full[data_source_column] != target_source_value]
            # print(f"\n数据来源不为 '{target_source_value}' 的记录共有 {len(df_other_sources)} 条。")


except FileNotFoundError:
    print(f"错误：文件 '{input_parquet_path}' 未找到。请检查文件路径是否正确。")
except KeyError:
    # 这个错误会在尝试访问一个不存在的列名时发生，虽然上面已经有检查，但作为备用
    print(f"错误：列名 '{data_source_column}' 在 Parquet 文件中未找到。请确认列名正确。")
except Exception as e:
    print(f"处理 Parquet 文件时发生错误：{e}")